# CatBoost 재검증

이전 실험에서 CatBoost는 시간 제약으로 3-fold만 검증했고(0.737, fold당 약
40~70초), 5-Fold를 끝까지 못 돌렸습니다. CatBoost는 LightGBM과 트리 구조가
다릅니다(대칭 트리, ordered boosting, 네이티브 카테고리 처리 방식이 다름).
제대로 5-Fold + 약간의 하이퍼파라미터 튜닝을 거치면 LightGBM과 다른 결과가
나올 가능성을 마지막으로 확인합니다.

**사전 준비**: `uv add catboost`

**예상 시간**: CatBoost는 fold당 30~70초 정도였으므로, Optuna 30 trial x
3-fold면 대략 30~60분 예상됩니다. 처음 5 trial 실행 시간을 보고 가늠하세요.


## 1. 라이브러리 및 설정

In [ ]:
import json
import os
import time

import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

t0 = time.time()
DATA_DIR = "../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]

N_TRIALS = 30
N_FOLDS_TUNING = 3
STUDY_DB = "sqlite:///optuna_catboost.db"
STUDY_NAME = "catboost_tuning_v1"
BEST_PARAMS_PATH = "best_params_catboost.json"

## 2. 데이터 로드 및 전처리 (main.py와 동일한 파이프라인, CatBoost 네이티브 카테고리 사용)

In [ ]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

# CatBoost는 문자열 카테고리를 네이티브로 처리하므로 LabelEncoder를 적용하지 않고
# cat_features 인덱스만 지정 (단, 결측 대체값 "해당없음"은 이미 문자열로 채워둔 상태)
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]
cat_feature_idx = [X.columns.get_loc(c) for c in cat_cols]

print(f"전처리 완료: {X.shape}, 카테고리 컬럼 {len(cat_cols)}개 (네이티브 처리)")

## 3. Optuna 탐색 (CatBoost 네이티브 카테고리)

In [ ]:
def make_objective(X, y):
    skf = StratifiedKFold(n_splits=N_FOLDS_TUNING, shuffle=True, random_state=1004)

    def objective(trial):
        params = {
            "random_state": 1004,
            "verbose": 0,
            "iterations": trial.suggest_int("iterations", 300, 1500),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "depth": trial.suggest_int("depth", 4, 10),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        }
        scores = []
        for tr_idx, val_idx in skf.split(X, y):
            model = CatBoostClassifier(cat_features=cat_feature_idx, **params)
            model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
            proba = model.predict_proba(X.iloc[val_idx])[:, 1]
            scores.append(roc_auc_score(y.iloc[val_idx], proba))
        return float(np.mean(scores))

    return objective


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=1004),
    storage=STUDY_DB,
    study_name=STUDY_NAME,
    load_if_exists=True,
)


def callback(study, trial):
    elapsed = time.time() - t0
    print(f"[{elapsed:6.1f}s] trial {trial.number:>3}: {trial.value:.5f}  (best: {study.best_value:.5f})", flush=True)
    with open(BEST_PARAMS_PATH, "w", encoding="utf-8") as f:
        json.dump({"best_value": study.best_value, "best_params": study.best_params}, f, ensure_ascii=False, indent=2)


print(f"Optuna 탐색 시작: {N_TRIALS} trials, {N_FOLDS_TUNING}-fold CV\n")
try:
    study.optimize(make_objective(X, y), n_trials=N_TRIALS, callbacks=[callback])
except KeyboardInterrupt:
    print("\n중단됨 - 지금까지 결과는 저장되어 있습니다.")

print(f"\nBest score (3-fold): {study.best_value:.5f}")
print(f"Best params: {study.best_params}")

## 4. 튜닝된 파라미터로 최종 5-Fold 학습

In [ ]:
with open(BEST_PARAMS_PATH, "r", encoding="utf-8") as f:
    tuned = json.load(f)
best_params = tuned["best_params"]
print(f"튜닝된 파라미터 로드 (3-fold 점수: {tuned['best_value']:.5f})")
print(best_params)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1004)
fold_scores = []
oof_catboost = np.zeros(len(y))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    model = CatBoostClassifier(cat_features=cat_feature_idx, random_state=1004, verbose=0, **best_params)
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    proba = model.predict_proba(X.iloc[val_idx])[:, 1]
    oof_catboost[val_idx] = proba
    score = roc_auc_score(y.iloc[val_idx], proba)
    fold_scores.append(score)
    print(f"[Fold {fold}/5] ROC-AUC: {score:.4f}")

catboost_score = np.mean(fold_scores)
print(f"\nCatBoost 5-Fold 평균: {catboost_score:.5f} (std: {np.std(fold_scores):.4f})")
print(f"비교 대상 (LightGBM 5-Fold, main.py): 0.7393")

np.save("catboost_oof.npy", oof_catboost)

## 5. 결론

CatBoost 튜닝 점수가 LightGBM(0.7393)을 능가하면, main.py에 모델을 교체하거나
LightGBM과 블렌딩(1번 노트북과 같은 방식)을 시도해볼 수 있습니다. 비슷하거나
낮다면, 트리 계열 모델 간 차이는 이 데이터에서 거의 없다는 기존 결론이
다시 확인되는 것입니다.
